# MCP-Server as a Tool

_Exposing an MCP Server as a tool to an agent for it to connect to it in a uniform and agent-agnostic way._

An agent connects to two MCP servers (math and weather) and selects the right MCP tool based on the user query and returns a final response.

**Prerequisites**

Refer to the following instructions to run the MCP servers first.

1. Open two seperate **Anaconda Prompt** terminals. 

2. Activate environment `agentic_ai` in each terminal using command `conda activate agentic_ai`.

3. In one terminal, run _math_ server by executing command `python 4_tool_calling_mcp_math_server.py` that listens on port 8001.

4. In the other terminal, run _weather_ server by executing command `python 4_tool_calling_mcp_weather_server.py` that listens on port 8002.

In [143]:
# Imports packages

from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langchain_ollama.chat_models import ChatOllama

In [ ]:
# Sets endpoints for Ollama models to be available over web requests.

OLLAMA_MODEL = "qwen3.5:2b"     # In this experiment, Qwen 3.5 2B is preferred over Llama 3.2 3b as the 
                                # latter one was observed being failed to prepare final response especially
                                # for questions on weather. Refer to model installation section in README.md.

OLLAMA_ENDPOINT = "http://localhost:11434/" 

## Tools


In [145]:
# A single MCP client that connects to both servers.

mcp_client = MultiServerMCPClient(
    {
        "math": {
            "url": "http://localhost:8001/mcp",
            "transport": "streamable_http",
        },
        "weather": {
            "url": "http://localhost:8002/mcp",
            "transport": "streamable_http",
        },
    }
)

In [146]:
TOOLS = await mcp_client.get_tools()    # Gets list of (LangChain) tools from both the servers.

display(TOOLS)                          # To inspect structure of tools

[StructuredTool(name='math', description='\n    Evaluate a basic arithmetic expression safely and return the result.\n    \n    Parameters:\n    expression (str): A string containing the arithmetic expression to evaluate.\n\n    Returns:\n    str: The result of the evaluated expression as a string.\n    ', args_schema={'properties': {'expression': {'title': 'Expression', 'type': 'string'}}, 'required': ['expression'], 'title': 'mathArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x7764dcd9f240>),
 StructuredTool(name='weather', description='\n    Returns weather information for a location.\n    It is a dummy implementation that always returns the same temperature for any location.\n    Production implementations would typically call an external weather API to get real data.\n    \n    Parameters:\n    location (str): The location for which to get the weather information.\n    \n    Retur

## Agent

In [ ]:
# Instruction to be used to set agent's behaviour

agent_instructions = """You are a simple assistant who can answer queries only on the following.
        1) Math expression evaluation
        2) Weather of a location (e.g. Mumbai, Bangalore, Delhi, Chennai and Delhi)

        Rules:
        - If a request is outside the above mentioned topics, politely decline and 
        inform user what you offer and let her re-enter the query.
        - For weather, extract location from the user queryand run the weather tool passing location as argument.
        - Do not call tools until all required fields are explicitly provided by the user.
        - For every new request, fetch parameter values from the user again. Do not reuse values from prior turns.

        For math expression evaluation:
        - Required field: expression
        - Extract math expression from the user query and call math(expression).

        For weather of a location:
        - Required fields: location
        - Extract location from the user query and call weather(location).
        - For now, weather tool is dummy and it returns same temperature for any location.

        For tools:
        - For tool calls, fill 'tool_calls' structure instead of JSON output in your response.

        For response
        - Prapare your response from tools messages, if exist.        
        """

In [148]:
# Initializes chat client
client = ChatOllama(model=OLLAMA_MODEL, 
                    reasoning=False,
                    base_url=OLLAMA_ENDPOINT
                    )

# Binds tools to the chat client
client_with_tools = client.bind_tools(tools=TOOLS)

In [ ]:
# Maintains a list of all messages that flow across interactions.
# The message list is first populated with SystemMessage that wraps
# intruction for model to behave in an expected way.

messages = [
    SystemMessage(content=agent_instructions)
    ]

In [ ]:
# User may provide one or multiple queries in single turn to be sent to client

query = "What is (2 + 3) * 4 - 5 / 2? And, what is the weather in Bangalore?"   # Two queries in one phrase

messages.append(HumanMessage(query))                # Appends user query (as a HumanMessage) into message list

In [151]:
# Passes messages (as a single input) to model and receives response [THIS MAY TAKE SEVERAL SECONDS]
client_message = client_with_tools.invoke(messages)

In [ ]:
# Displays the (structured) model-returned message to inspect element `tool_calls` to check
# if model has returned the correct tool name(s) with argument(s) for client to call those tools later
display(client_message)

AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.5:2b', 'created_at': '2026-06-07T16:58:13.226708183Z', 'done': True, 'done_reason': 'stop', 'total_duration': 47777162598, 'load_duration': 6930592468, 'prompt_eval_count': 752, 'prompt_eval_duration': 25713103554, 'eval_count': 67, 'eval_duration': 14784292059, 'logprobs': None, 'model_name': 'qwen3.5:2b', 'model_provider': 'ollama'}, id='lc_run--019ea304-8ec4-7cc3-b5f0-797c8f8e2673-0', tool_calls=[{'name': 'math', 'args': {'expression': '(2 + 3) * 4 - 5 / 2'}, 'id': '6d3012ca-6442-4e29-ab40-6e9b60d816d9', 'type': 'tool_call'}, {'name': 'weather', 'args': {'location': 'Bangalore'}, 'id': '2df2136e-0aeb-46fe-8257-8ab95a8daa3d', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 752, 'output_tokens': 67, 'total_tokens': 819})

In [153]:
messages.append(client_message)     # Appends last received message into message list

In [154]:
# Now, client loops over each tool name to call the tool manually for execution

for tool_call in client_message.tool_calls:
    print(f"Tool call (detailed): {tool_call}")
    tool_name = tool_call["name"].lower()               # Extracts tool name and converts to lower case
    selected_tool = next(tool for tool in TOOLS if tool.name == tool_name)
    print(f"Selected tool: {selected_tool}")
  
    raw_tool_message = await selected_tool.ainvoke(tool_call.get("args"))

    # Normalizes MCP tool output to text
    if isinstance(raw_tool_message, list):
        content = "\n".join(
            item.get("text", str(item)) if isinstance(item, dict) else str(item) for item in raw_tool_message
        )
    else:
        content = str(raw_tool_message)

    print(f"Arguments: {tool_call["args"]}, \nTool raw message: {raw_tool_message}, \nNormalized output: {content}\n")
    
    # Message received from tool gets added to message list
    messages.append(
        ToolMessage(
            content=content,
            tool_call_id=tool_call["id"],
            name=tool_call["name"],
        )
    )

Tool call (detailed): {'name': 'math', 'args': {'expression': '(2 + 3) * 4 - 5 / 2'}, 'id': '6d3012ca-6442-4e29-ab40-6e9b60d816d9', 'type': 'tool_call'}
Selected tool: name='math' description='\n    Evaluate a basic arithmetic expression safely and return the result.\n    \n    Parameters:\n    expression (str): A string containing the arithmetic expression to evaluate.\n\n    Returns:\n    str: The result of the evaluated expression as a string.\n    ' args_schema={'properties': {'expression': {'title': 'Expression', 'type': 'string'}}, 'required': ['expression'], 'title': 'mathArguments', 'type': 'object'} response_format='content_and_artifact' coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x7764dcd9f240>
Arguments: {'expression': '(2 + 3) * 4 - 5 / 2'}, 
Tool raw message: [{'type': 'text', 'text': '17.5', 'id': 'lc_b8887a0f-3544-4771-b0ed-fee950c3f332'}], 
Normalized output: 17.5

Tool call (detailed): {'name': 'weather', 'args': {'location': 'Bangalor

In [155]:
# Displays the final message list for analysis
display(messages)

[SystemMessage(content="You are a simple assistant who can answer queries only on the following.\n        1) Math expression evaluation\n        2) weather of a location (e.g. Mumbai, Bangalore, Delhi, Chennai and Delhi)\n\n        Rules:\n        - If a request is outside the above mentioned topics, politely decline and \n        inform user what you offer and let her re-enter the query.\n        - For weather, extract location from the user queryand run the weather tool passing location as argument.\n        - Do not call tools until all required fields are explicitly provided by the user.\n        - For every new request, fetch parameter values from the user again. Do not reuse values from prior turns.\n\n        For math expression evaluation:\n        - Required field: expression\n        - Extract math expression from the user query and call math(expression).\n\n        For weather of a location:\n        - Required fields: location\n        - Extract location from the user query

In [156]:
final_response = client_with_tools.invoke(messages)         # Passes all messages back to model to generate final response
print(f"Final Agent Response: {final_response.content}")    # Prints final response that the user would see

Final Agent Response: The result of the expression `(2 + 3) * 4 - 5 / 2` is 17.5.

The current temperature in Bangalore is 28°C.
